# Milestone 4 — Training and loading pretrained weights

This notebook trains a compact instance of the same Llama architecture on toy data, then loads the official full Llama 3.2 1B checkpoint. Select **Runtime → Change runtime type → T4 GPU**. Add a Colab secret named `HF_TOKEN`, enable notebook access, and never paste the token into a cell.

In [ ]:
%pip install -q 'tiktoken>=0.7.0,<1' 'huggingface-hub>=0.26.0,<1' 'safetensors>=0.4.1,<1'

In [ ]:
import os
import subprocess

repo_dir = '/content/debug-diary-1-build-your-first-llm-from-scratch-lp'
repo_url = 'https://github.com/manning-lp/debug-diary-1-build-your-first-llm-from-scratch-lp.git'
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)
else:
    subprocess.run(['git', '-C', repo_dir, 'pull', '--ff-only'], check=True)
os.chdir(repo_dir)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import gc
import torch
from google.colab import userdata

from llama3 import LLAMA32_CONFIG, Llama3Model
from milestone_2 import prepare_model_config
from milestone_4 import (
    download_pretrained_weights,
    generate,
    load_pretrained_weights,
    prepare_toy_sequences,
    train_on_toy_data,
)
from tokenizer import Tokenizer, download_tokenizer_model

if not torch.cuda.is_available():
    raise RuntimeError('Select a T4 GPU from Runtime → Change runtime type, then rerun.')
device = torch.device('cuda')
hf_token = userdata.get('HF_TOKEN')
print('Device:', torch.cuda.get_device_name(0))

In [ ]:
model_path = download_tokenizer_model(token=hf_token, local_dir='/content/Llama-3.2-1B')
tokenizer = Tokenizer(model_path)
print(f'Loaded tokenizer with {tokenizer.model.n_vocab:,} tokens')

## Generate and train on toy data

The compact model below uses the production tokenizer vocabulary and the exact same architecture, but smaller hidden dimensions so AdamW training fits comfortably on a Colab T4.

In [ ]:
torch.manual_seed(7)
toy_config = {
    'vocab_size': tokenizer.model.n_vocab,
    'context_length': 96,
    'emb_dim': 128,
    'n_heads': 4,
    'n_layers': 2,
    'hidden_dim': 384,
    'n_kv_groups': 2,
    'rope_base': 500_000.0,
    'dtype': torch.float32,
    'rope_freq': None,
}
toy_model = Llama3Model(toy_config).to(device)
prompt = 'The Eiffel Tower is'
prompt_ids = torch.tensor([tokenizer.encode(prompt, bos=True)], device=device)
before_ids = generate(
    toy_model, prompt_ids, max_new_tokens=12,
    context_size=toy_config['context_length'], temperature=0.0,
)
before_text = tokenizer.decode(before_ids[0].tolist())
print('Before toy training:', before_text)

In [ ]:
toy_texts = [
    'The Eiffel Tower is located in Paris, France.',
    'The Eiffel Tower was completed in 1889.',
    'The capital of France is Paris.',
    'Paris is famous for the Eiffel Tower.',
]
toy_sequences = prepare_toy_sequences(tokenizer, toy_texts, device=device)
losses = train_on_toy_data(
    toy_model, toy_sequences, epochs=8, learning_rate=1e-3
)
assert losses[-1] < losses[0], 'Toy-training loss did not decrease'
print(f'Loss decreased from {losses[0]:.4f} to {losses[-1]:.4f}')

In [ ]:
after_ids = generate(
    toy_model, prompt_ids, max_new_tokens=16,
    context_size=toy_config['context_length'], temperature=0.0,
)
after_text = tokenizer.decode(after_ids[0].tolist())
print('After toy training:', after_text)
del toy_model, toy_sequences, prompt_ids, before_ids, after_ids
gc.collect()
torch.cuda.empty_cache()

## Load the official pretrained Llama 3.2 1B checkpoint

This downloads the gated checkpoint only into the Colab runtime, maps every learned tensor into our from-scratch model, and verifies whether the output head uses weight tying.

In [ ]:
weights = download_pretrained_weights(
    token=hf_token, local_dir='/content/Llama-3.2-1B'
)
pretrained_config = prepare_model_config(
    LLAMA32_CONFIG, context_length=1024, device=device
)
pretrained_model = Llama3Model(pretrained_config)
uses_weight_tying = load_pretrained_weights(
    pretrained_model, pretrained_config, weights
)
del weights
gc.collect()
pretrained_model.to(device).eval()
print('Pretrained weights loaded successfully.')
print('Weight tying:', uses_weight_tying)

In [ ]:
prompt = 'The Eiffel Tower is'
input_ids = torch.tensor([tokenizer.encode(prompt, bos=True)], device=device)
generated_ids = generate(
    pretrained_model,
    input_ids,
    max_new_tokens=30,
    context_size=pretrained_config['context_length'],
    temperature=0.0,
    eos_id=tokenizer.eos_id,
)
generated_text = tokenizer.decode(generated_ids[0].tolist())
print('Pretrained model output:')
print(generated_text)
assert generated_ids.shape[1] > input_ids.shape[1]
print('Milestone 4 validation passed.')